In [1]:
import onnxruntime as ort
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score, roc_auc_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

In [2]:
class CWRUDataset(Dataset):
    def __init__(self, path):
        self.dataset = torch.load(path)
        self.data    = self.dataset['data']
        self.label   = self.dataset['label']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image = self.data[index].float() / 255.0
        label = self.label[index].long()
        return image, label

class ONNXModel:
    def __init__(self, model_path):
        self.session = ort.InferenceSession(model_path)
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
    
    def predict(self, x):
        outputs = self.session.run([self.output_name], {self.input_name: x})
        return outputs[0]
    
scal_path = r"D:\Capstone\dataset\processed\CNN\CWRU\Scalogram\test.pt"
spec_path = r"D:\Capstone\dataset\processed\CNN\CWRU\Spectrogram\test.pt"

In [29]:
scal_mobilenetv4 = r"D:\Capstone\models\ONNX\CNN\SCALOGRAM\MobileNetV4\best_model_int8.onnx"
spec_mobilenetv4 = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\MobileNetV4\best_model_int8.onnx"

scal_ghostnetv3 = r"D:\Capstone\models\ONNX\CNN\SCALOGRAM\GhostNetV3\best_model_int8.onnx"
spec_ghostnetv3 = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\GhostNetV3\best_model_int8.onnx"

scal_edgenext = r"D:\Capstone\models\ONNX\CNN\SCALOGRAM\EdgenextXXS\best_model_int8.onnx"
spec_edgenext = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\EdgenextXXS\best_model_int8.onnx"

scal_tinynetd = r"D:\Capstone\models\ONNX\CNN\SCALOGRAM\TinyNetD\best_model_int8.onnx"
spec_tinynetd = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\TinyNetD\best_model_int8.onnx"

In [30]:
def onnx_eval(model_path, eval_path):
    onnx_model = ONNXModel(model_path)
    data_loader = DataLoader(CWRUDataset(eval_path), batch_size=256, shuffle=True)
    y_true, y_pred, all_probs = [], [], []

    for img, y in data_loader:
        probs = onnx_model.predict(np.array(img))
        logits = np.argmax(probs, axis=1)

        y_true.extend(y.numpy())
        y_pred.extend(logits)
        all_probs.extend(probs)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy}")

    precision = precision_score(y_true, y_pred, average='macro')
    print()
    print(f"Precision: {precision}")

    f1 = f1_score(y_true, y_pred, average='macro')
    print()
    print(f"F1: {f1}")

    recall = recall_score(y_true, y_pred, average='macro')
    print()
    print(f"Recall: {recall}")

    num_classes = all_probs.shape[1]
    labels_bin = label_binarize(y_true, classes=list(range(num_classes)))

    auc_roc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')
    print()
    print(f"AUC-ROC {auc_roc}")

    k = cohen_kappa_score(y_true, y_pred)
    print()
    print(f"K: {k}")

    mcc = matthews_corrcoef(y_true, y_pred)
    print()
    print(f"MCC: {mcc}")

    cm = confusion_matrix(y_true, y_pred)
    print()
    print('Confusion Matrix')
    print(cm)

In [31]:
onnx_eval(scal_mobilenetv4, scal_path)

Accuracy: 0.9902083856389656

Precision: 0.9912760622847523

F1: 0.9915239664120195

Recall: 0.9917727677524585

AUC-ROC 0.9993642959703752

K: 0.986122731064251

MCC: 0.9861230092630151

Confusion Matrix
[[ 466    0    0    0]
 [   1 1624    4   14]
 [   0    5  931    1]
 [   0   12    2  923]]


In [36]:
onnx_eval(spec_mobilenetv4, spec_path)

Accuracy: 0.9703740898819985

Precision: 0.9784730421263437

F1: 0.9726977697520099

Recall: 0.9687458387220191

AUC-ROC 0.9984906075035597

K: 0.9577922970138297

MCC: 0.9585316577925639

Confusion Matrix
[[ 466    0    0    0]
 [   0 1641    2    0]
 [   0   76  831   30]
 [   0    6    4  927]]


In [33]:
onnx_eval(scal_ghostnetv3, scal_path)

Accuracy: 0.9688676876726086

Precision: 0.9773263137218272

F1: 0.972147378008906

Recall: 0.9684061160474469

AUC-ROC 0.9944781582070834

K: 0.9556630089926821

MCC: 0.9562611396544651

Confusion Matrix
[[ 466    0    0    0]
 [   0 1630   13    0]
 [   0    4  924    9]
 [   0   83   15  839]]


In [37]:
onnx_eval(spec_ghostnetv3, spec_path)

Accuracy: 0.9199096158674366

Precision: 0.929688948694515

F1: 0.9273107754247844

Recall: 0.9330023689648071

AUC-ROC 0.9949663104638226

K: 0.8874517303037406

MCC: 0.8908966662853648

Confusion Matrix
[[ 466    0    0    0]
 [   0 1485  121   37]
 [   0   16  918    3]
 [   0    4  138  795]]


In [38]:
onnx_eval(scal_edgenext, scal_path)

Accuracy: 0.8066783831282952

Precision: 0.8221666036850835

F1: 0.7886419403268623

Recall: 0.7975379524790986

AUC-ROC 0.9420390648258311

K: 0.7191094693751646

MCC: 0.7340480532855379

Confusion Matrix
[[ 466    0    0    0]
 [   0 1617   26    0]
 [   0  397  306  234]
 [   0   16   97  824]]


In [39]:
onnx_eval(spec_edgenext, spec_path)

Accuracy: 0.9158925433090636

Precision: 0.9375501185586035

F1: 0.9212306626305737

Recall: 0.9108482933644952

AUC-ROC 0.9702065488995169

K: 0.8789992752498176

MCC: 0.8824359145047238

Confusion Matrix
[[ 466    0    0    0]
 [   0 1641    2    0]
 [   0  234  703    0]
 [   0    1   98  838]]


In [40]:
onnx_eval(scal_tinynetd, scal_path)

Accuracy: 0.9691187547075069

Precision: 0.9797851984445405

F1: 0.9729382172254838

Recall: 0.9681739567241112

AUC-ROC 0.9832223111508536

K: 0.9559712660046091

MCC: 0.9568430017971957

Confusion Matrix
[[ 465    0    0    1]
 [   0 1632    0   11]
 [   0   97  831    9]
 [   0    5    0  932]]


In [41]:
onnx_eval(spec_tinynetd, spec_path)

Accuracy: 0.9384885764499121

Precision: 0.9470016844188185

F1: 0.937020391802273

Recall: 0.9369247692906292

AUC-ROC 0.9770859464657964

K: 0.9127635928095749

MCC: 0.915823930273216

Confusion Matrix
[[ 466    0    0    0]
 [   0 1623   16    4]
 [   0   22  914    1]
 [   0    6  196  735]]
